# Cuaderno de conexiones entre nodos
## Objetivo: Una vez obtenidos los nodos se creará un csv que conecte los nodos de los hospitales y los nodos totales

In [3]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

In [5]:
hospitales = pd.read_csv("hospitales_limpio.csv")
nodos_viales = pd.read_csv("nodos_viales.csv")
rutas_viales = pd.read_csv("rutas_viales.csv")

# Función para calcular distancia entre coordenadas (considerando Harvine pues no estamos sobre un plano)
def distancia_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

# Crear conexiones hospital -> nodo vial más cercano
conexiones_hospitales = []

for _, hospital in hospitales.iterrows():

    distancias = []

    for _, nodo in nodos_viales.iterrows():

        d = distancia_km(
            hospital["lat"], hospital["lon"],
            nodo["lat"], nodo["lon"]
        )

        distancias.append((nodo["id_nodo"], d))

    cercanos = sorted(distancias, key=lambda x: x[1])[:3]

    for id_nodo, d in cercanos:

        conexiones_hospitales.append({
            "origen": hospital["id_osm"],
            "destino": id_nodo,
            "distancia_km": round(d, 3),
            "factor_trafico": 1.0,
            "peso": round(d, 3),
            "tipo_ruta": "hospital-vial"
        })

conexiones_hospitales = pd.DataFrame(conexiones_hospitales)
#Al final se generan las conexiones entre hospitales y 3 nodos más cercanos 

In [7]:
# Aquí agregaremos conexiones extra entre nodos viales.
# Esto ayuda a que Dijkstra tenga más alternativas
# y evita zonas desconectadas.
nuevas_rutas = []

for _, origen in nodos_viales.iterrows():

    distancias = []

    for _, destino in nodos_viales.iterrows():

        if origen["id_nodo"] != destino["id_nodo"]:

            d = distancia_km(
                origen["lat"], origen["lon"],
                destino["lat"], destino["lon"]
            )

            distancias.append((destino["id_nodo"], d))

    cercanos = sorted(distancias, key=lambda x: x[1])[:3] #3 nodos mas cercanos

    for id_destino, d in cercanos:

        nuevas_rutas.append({
            "origen": origen["id_nodo"],
            "destino": id_destino,
            "distancia_km": round(d, 3),
            "factor_trafico": 1.0,
            "peso": round(d, 3),
            "tipo_ruta": "conexion-vial-cercana"
        })

rutas_extra = pd.DataFrame(nuevas_rutas)

# Tipos de ruta
rutas_viales["tipo_ruta"] = "vial-vial"
conexiones_hospitales["tipo_ruta"] = "hospital-vial"

# Grafo final conectado
rutas_completas = pd.concat(
    [
        rutas_viales,
        conexiones_hospitales,
        rutas_extra
    ],
    ignore_index=True
)

# Guardar
rutas_completas.to_csv("rutas_completas.csv", index=False)
rutas_completas.to_excel("rutas_completas.xlsx", index=False)

rutas_completas.head()

,origen,destino,distancia_km,factor_trafico,peso,tipo_ruta
0,V0_0,V0_1,0.376,1.0,0.376,vial-vial
1,V0_1,V0_2,0.204,1.0,0.204,vial-vial
2,V1_0,V1_1,0.332,1.0,0.332,vial-vial
3,V1_1,V1_2,0.064,1.0,0.064,vial-vial
4,V2_0,V2_1,0.109,1.0,0.109,vial-vial
